In [7]:
%%sh 
which python

/home/aadi/projects/f1-finishing-position/.venv/bin/python


In [179]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
import category_encoders as ce

pd.set_option("display.max_columns", None)

DATA_PATH = Path("../data")

df = pd.DataFrame()
for year in range(2020, 2025, 1):
    df_year = pd.read_parquet(DATA_PATH / f"transformed_data_{year}.parquet")
    df = pd.concat([df, df_year], axis=0)


df.rename(columns={"target": "end_position"}, inplace=True)


In [182]:
df = (
    df.assign(pit_1_to_total=lambda x: x.pit_1_lap / df.total_laps)
    .assign(pit_2_to_total=lambda x: x.pit_2_lap / df.total_laps)
    .assign(pit_3_to_total=lambda x: x.pit_3_lap / df.total_laps)
    .assign(pit_4_to_total=lambda x: x.pit_4_lap / df.total_laps)
    .assign(pit_5_to_total=lambda x: x.pit_5_lap / df.total_laps)
    .assign(pit_6_to_total=lambda x: x.pit_6_lap / df.total_laps)
    .assign(pit_7_to_total=lambda x: x.pit_7_lap / df.total_laps)
    .assign(more_than_2_pit_stops=lambda x: x.pit_3_lap.notna().astype(bool))
    .assign(won_race=lambda x: x.end_position== 1)
)

In [183]:
df = df.drop(['Driver','DriverNumber'], axis=1)

In [184]:
df.to_parquet(DATA_PATH / "final_data.parquet", index=False)

In [140]:

X_train, X_test, y_train, y_test = train_test_split(df.drop(['target', 'won_race'], axis=1), df['won_race'].astype(int), test_size=0.2, random_state=42)

In [167]:
feature_pipeline = Pipeline(
    steps=[
        (
            "quantile_e",
            ce.QuantileEncoder(
                verbose=0,
                cols=[
                    # "pit_1_to_total",
                    # "pit_2_to_total",
                    # "pit_3_to_total",
                    # "pit_4_to_total",
                    # "pit_5_to_total",
                    # "pit_6_to_total",
                    # "pit_7_to_total",
                    "pit_1_intime",
                    "pit_2_intime",
                    "pit_3_intime",
                    "pit_4_intime",
                    "pit_5_intime",
                    "pit_6_intime",
                    "pit_7_intime",
                    # "pit_1_duration",
                    # "pit_2_duration",
                    # "pit_3_duration",
                    # "pit_4_duration",
                    # "pit_5_duration",
                    # "pit_6_duration",
                    # "pit_7_duration",
                ],
                return_df=True,
            ),
        ),
        (
            "target_e",
            ce.TargetEncoder(
                verbose=0,
                cols=[
                    "pit_1_tyre_before", "pit_2_tyre_before", "pit_3_tyre_before",
                    "pit_4_tyre_before", "pit_5_tyre_before", "pit_6_tyre_before", "pit_7_tyre_before",
                    "pit_1_tyre_after", "pit_2_tyre_after", "pit_3_tyre_after",
                    "pit_4_tyre_after", "pit_5_tyre_after", "pit_6_tyre_after",
                    ],
                handle_missing='return_nan',
                return_df=True,
            ),
        ),
        (
            "one_hot_e",
            ce.OneHotEncoder(
                verbose=0,
                cols=[
                    # "pit_1_fresh",
                    "pit_2_fresh",
                    "pit_3_fresh",
                    "pit_4_fresh",
                    "pit_5_fresh",
                    "pit_6_fresh",
                    "pit_7_fresh",
                ],
                use_cat_names=True,
                handle_missing='return_nan',
                return_df=True,
            ),
        ),
    ]
)
feature_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('quantile_e', ...), ('target_e', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,verbose,0
,cols,"['pit_1_intime', 'pit_2_intime', ...]"
,drop_invariant,False
,return_df,True
,handle_missing,'value'
,handle_unknown,'value'
,quantile,0.5


In [168]:
feature_pipeline.fit(X_train, y_train)
X_train_enc = feature_pipeline.fit_transform(X_train, y_train)

In [169]:

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)

In [170]:
y_train.value_counts()

won_race
0    1465
1      77
Name: count, dtype: int64

In [171]:
from xgboost import XGBClassifier


xgb = XGBClassifier(enable_categorical=True)

xgb.fit(X_train_enc, y_train_enc)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [172]:
y_train_enc

array([0, 1, 0, ..., 0, 0, 0], shape=(1542,))

In [173]:
pd.DataFrame({'feature': xgb.feature_names_in_, 'feature_importance': xgb.feature_importances_}).sort_values(by='feature_importance', ascending=False)

,feature,feature_importance
7,pit_1_intime,0.997352
37,starting_position,0.002448
26,WindDirection,0.000200
0,num_pit_stops,0.000000
3,pit_3_lap,0.000000
...,...,...
67,pit_4_to_total,0.000000
68,pit_5_to_total,0.000000
69,pit_6_to_total,0.000000
70,pit_7_to_total,0.000000


In [178]:
from sklearn.metrics import classification_report


y_test_enc = le.transform(y_test)
X_test_enc = feature_pipeline.transform(X_test)
print(classification_report(y_test_enc, xgb.predict(X_test_enc)))

              precision    recall  f1-score   support

           0       0.94      1.00      0.97       361
           1       1.00      0.04      0.08        25

    accuracy                           0.94       386
   macro avg       0.97      0.52      0.52       386
weighted avg       0.94      0.94      0.91       386

